# Value Iteration Policies

Run classic Value Iteration on the supported MDPs and serialize the resulting optimal policies under `policies/`.

In [1]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from algorithms.value_iteration.value_iteration import ValueIteration
from mdp.gridworld.gridworld import GridWorldMDP
from mdp.cats_v_monsters.cats_vs_monsters import CatsVMonstersMDP


In [2]:
@dataclass(slots=True)
class EnvSpec:
    name: str
    builder: callable
    description: str

ENV_REGISTRY = {
    'gridworld': EnvSpec(
        name='GridWorld 5x5',
        builder=lambda: GridWorldMDP(rows=5, cols=5),
        description='Stochastic 5x5 grid with terminal goal.'
    ),
    'cats_vs_monsters': EnvSpec(
        name='Cats vs Monsters',
        builder=lambda: CatsVMonstersMDP(rows=5, cols=5),
        description='Grid with tasty food, monsters, and obstacles.'
    ),
}

POLICY_DIR = Path('policies')
POLICY_DIR.mkdir(parents=True, exist_ok=True)

ARROW_MAP = {
    'up': '↑',
    'down': '↓',
    'left': '←',
    'right': '→',
}


In [3]:
def _format_policy_grid(policy: dict):
    tuple_states = [
        state for state in policy.keys()
        if isinstance(state, tuple)
        and len(state) == 2
        and all(isinstance(x, int) for x in state)
    ]
    if not tuple_states:
        return None
    max_r = max(r for r, _ in tuple_states)
    max_c = max(c for _, c in tuple_states)
    grid = [['·' for _ in range(max_c + 1)] for _ in range(max_r + 1)]
    for state in tuple_states:
        r, c = state
        action = policy.get(state)
        grid[r][c] = ARROW_MAP.get(action, action or '·')
    lines = ['Policy grid (rows top→bottom, cols left→right):']
    for row in grid:
        lines.append(' '.join(row))
    return '\n'.join(lines)


def save_optimal_policy(env_key, *, gamma=0.99, delta=1e-4):
    if env_key not in ENV_REGISTRY:
        raise KeyError(f'Unknown environment: {env_key}')
    spec = ENV_REGISTRY[env_key]
    mdp = spec.builder()
    solver = ValueIteration(mdp)
    history = solver.run_value_iteration(gamma=gamma, delta=delta)
    final_iter = max(history.keys())
    value_function, policy = history[final_iter]

    lines = [
        f'Optimal policy via Value Iteration',
        f'Environment: {spec.name} ({env_key})',
        f'Iterations: {final_iter}',
        f'Gamma={gamma}, Delta={delta}',
        '',
        'state -> best action',
    ]
    for state in sorted(policy.keys()):
        lines.append(f'{state} -> {policy[state]}')

    grid = _format_policy_grid(policy)
    if grid:
        lines.extend(['', grid])

    out_path = POLICY_DIR / f'{env_key}_optimal_policy.txt'
    out_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')
    return {
        'path': out_path,
        'iterations': final_iter,
        'value_function': value_function,
        'policy': policy,
    }



In [ ]:
save_optimal_policy('gridworld')
save_optimal_policy('cats_vs_monsters')

{'path': PosixPath('policies/cats_vs_monsters_optimal_policy.txt'),
 'iterations': 109,
 'value_function': {(0, 0): 7.044029807539143,
  (0, 1): 7.1942684308105616,
  (0, 2): 7.1084077319718615,
  (0, 3): 7.185667090388896,
  (0, 4): 8.148200821838818,
  (1, 0): 7.181626175374084,
  (1, 1): 7.410734247208789,
  (1, 2): 7.626954133458248,
  (1, 3): 7.898076064865144,
  (1, 4): 9.254996005018445,
  (2, 0): 7.008314174105598,
  (2, 1): 0.0,
  (2, 2): 0.0,
  (2, 3): 0.0,
  (2, 4): 9.693313448124021,
  (3, 0): 6.684032543602336,
  (3, 1): 5.776190059099976,
  (3, 2): 0.0,
  (3, 3): 9.723963895117988,
  (3, 4): 9.905338173205179,
  (4, 0): 5.776190059099977,
  (4, 1): 7.2069398589122144,
  (4, 2): 9.693313448124021,
  (4, 3): 9.905338173205179,
  (4, 4): 0.0},
 'policy': {(0, 0): 'right',
  (0, 1): 'down',
  (0, 2): 'left',
  (0, 3): 'down',
  (0, 4): 'right',
  (1, 0): 'right',
  (1, 1): 'right',
  (1, 2): 'right',
  (1, 3): 'down',
  (1, 4): 'down',
  (2, 0): 'up',
  (2, 1): None,
  (2, 2)